# 01-Component / demo — Gradio x HuggingFace Pipeline 互動示範 (2026)

## 學習目標

1. 掌握以 `pipeline()` 搭配 `device_map='auto'` 與 `torch_dtype=torch.bfloat16` 載入模型的 2026 統一慣例。
2. 了解 `gr.Interface.from_pipeline()` 如何將 HuggingFace pipeline 零程式碼包成互動 UI。
3. 實際體驗文本分類 (text-classification) 與抽取式閱讀理解 (question-answering) 兩條 pipeline。

## 前置知識

- 已完成 `01-Component/01pipeline/01.pipeline.ipynb`（pipeline 基礎）。

## 銜接地圖

```
01pipeline → [本 notebook: demo] → 02tokenizer → 03Model → 04Datasets → 05evaluate → 06Trainer
```

本 notebook 是「01-Component」的整合示範壓軸，用 Gradio 把前面學到的 pipeline 包成可互動的 web UI，驗證你的理解。

## 環境鎖版本

**為什麼要鎖版本？**

HuggingFace 生態系更新頻繁，`transformers 4.42+` 棄用了裸 `load_in_4bit/8bit` kwargs；`gradio 5.x` 重寫了 component API。鎖版本是讓教學 notebook 半年後還能跑的最低成本方式。

> 若在 Colab / Lightning Studio 執行，執行下方 cell 安裝套件後需 **重啟 Runtime**。

In [ ]:
# Install pinned dependencies — run once, then restart kernel
%pip install --quiet \
    "transformers>=4.46,<5" \
    "torch>=2.4" \
    "accelerate>=1.0" \
    "safetensors>=0.4" \
    "sentencepiece>=0.2" \
    "gradio>=5.0,<6"

## 匯入套件

明確列舉需要的類別，避免萬用匯入（`from transformers import *`）污染命名空間與影響 IDE 補全。

In [ ]:
import torch
import gradio as gr
from transformers import pipeline

print(f"torch      : {torch.__version__}")
print(f"gradio     : {gr.__version__}")
print(f"cuda avail : {torch.cuda.is_available()}")

## 1. 為何用 `device_map='auto'` 與 `torch_dtype=torch.bfloat16`

### dtype 選擇

| dtype | 精度 | VRAM 占用 | 適合場景 |
|---|---|---|---|
| `float32` | 最高 | 4 bytes/param | 純 CPU 推論、精度敏感研究 |
| `float16` | 中 | 2 bytes/param | 舊 GPU（Volta、Tesla）—— 數值範圍窄，易 overflow |
| `bfloat16` | 中 | 2 bytes/param | **推薦**：Ampere+ GPU、Apple Silicon —— 範圍與 fp32 相同，訓練穩定 |

### `device_map='auto'`

`accelerate` 會依照目前硬體（GPU → CPU → disk）自動分配層，  
- 有 GPU：模型全放 GPU  
- GPU VRAM 不足：自動 CPU/disk offload  
- 無 GPU：自動退回 CPU  

這讓同一份程式碼在 Colab Free（T4）、A100 與純 CPU 環境都能執行，不需要手動寫 `.cuda()` 或 `device=0`。

### `use_safetensors=True`

- safetensors 格式不含 Python pickle，**無法執行任意程式碼**，免疫 pickle 反序列化漏洞。
- 載入速度比 `.bin` 快 2–5 倍（mmap 零拷貝）。
- 2026 年 HuggingFace Hub 上絕大多數模型已同時提供兩種格式；`use_safetensors=True` 優先選 safetensors。

## 2. 文本分類 — Gradio 互動示範

### 任務說明

`text-classification` pipeline 將一段文字映射至預定義的類別（例如：正面 / 負面情感）。

本示範使用 `uer/roberta-base-finetuned-dianping-chinese`，這是在大眾點評中文評論語料上微調的 RoBERTa，輸出「正面/負面」情感標籤。

`gr.Interface.from_pipeline()` 會自動偵測 pipeline 的輸入型態（文字）和輸出型態（label + score），產生對應的輸入框與輸出欄位，**不需要手寫任何 Gradio component**。

> **VRAM 需求**：RoBERTa-base（125M 參數）以 bfloat16 載入約需 **0.25 GB VRAM**，在 Colab Free 的 T4 上完全沒問題。CPU 執行亦可，速度稍慢。

In [ ]:
# Text classification demo with 2026 pipeline convention
text_clf_pipeline = pipeline(
    "text-classification",
    model="uer/roberta-base-finetuned-dianping-chinese",
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

demo_clf = gr.Interface.from_pipeline(
    text_clf_pipeline,
    title="中文情感分類 (RoBERTa)",
    description="輸入一段中文評論，模型會預測正面或負面情感。",
    examples=[
        ["這家餐廳的菜色非常美味，服務也很周到，下次還會再來！"],
        ["等了一個小時才上菜，食物也冷掉了，非常失望。"],
    ],
)
demo_clf.launch()

## 3. 抽取式閱讀理解 — Gradio 互動示範

### 任務說明

`question-answering` pipeline 屬於「抽取式 QA」（Extractive QA）：  
給定一段**上下文 (context)** 與一個**問題 (question)**，模型從上下文中抽取一段連續文字作為答案，不會生成新文字。

這與生成式 QA（LLM 直接生成答案）不同：

| 類型 | 模型代表 | 用途 |
|---|---|---|
| 抽取式 QA | BERT / RoBERTa | 法律文件、合約審閱（答案必須原文引用） |
| 生成式 QA | GPT / Llama | 開放問答、摘要式回答 |

### 模型說明

`uer/roberta-base-chinese-extractive-qa` 是以 SQuAD 格式中文數據微調的 RoBERTa，輸出 `answer`、`start`、`end`、`score` 四個欄位。

### `from_pipeline()` 自動 UI 生成

Gradio 偵測到 `question-answering` pipeline 後，會自動建立兩個輸入框（question、context）與結果顯示欄位，對應 pipeline 的標準輸入格式：

```python
pipeline("question-answering")({
    "question": "...",
    "context": "..."
})
```

> **VRAM 需求**：同上，RoBERTa-base 以 bfloat16 約 **0.25 GB**。

In [ ]:
# Extractive QA demo with 2026 pipeline convention
qa_pipeline = pipeline(
    "question-answering",
    model="uer/roberta-base-chinese-extractive-qa",
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

demo_qa = gr.Interface.from_pipeline(
    qa_pipeline,
    title="中文抽取式閱讀理解 (RoBERTa)",
    description="輸入上下文段落與問題，模型從段落中找出答案片段。",
    examples=[
        [
            "台灣的首都是哪裡？",
            "台灣是位於亞洲東部的島嶼，首都為台北市，面積約36,000平方公里，人口超過2,300萬人。",
        ],
        [
            "HuggingFace 的總部在哪個城市？",
            "HuggingFace 是一家成立於2016年的 AI 公司，總部位於美國紐約，致力於開源 NLP 工具與模型。",
        ],
    ],
)
demo_qa.launch()

## 4. 進階：同時啟動多個 Demo（TabbedInterface）

如果想把多個 pipeline 整合在同一個網頁的不同分頁，可以用 `gr.TabbedInterface`，不必分開呼叫 `launch()`。

In [ ]:
# Combine both demos into one tabbed UI — optional, comment out if running inline
tabbed = gr.TabbedInterface(
    interface_list=[demo_clf, demo_qa],
    tab_names=["文本分類", "閱讀理解"],
    title="HuggingFace Pipeline 互動示範 (2026)",
)
tabbed.launch()

## 小結

| 知識點 | 核心概念 |
|---|---|
| `pipeline(device_map='auto', torch_dtype=torch.bfloat16)` | 2026 統一載入慣例，跨硬體可攜、節省 VRAM |
| `use_safetensors=True` | 安全、快速的權重格式（無 pickle 風險）|
| `gr.Interface.from_pipeline()` | 零程式碼把 pipeline 包成 Web UI |
| `gr.TabbedInterface` | 多 Demo 整合到單一介面 |
| 抽取式 vs 生成式 QA | 前者從原文抽段落，後者生成新文字 |

## 練習題

1. 把 `text-classification` 的模型換成 `lxyuan/distilbert-base-multilingual-cased-sentiments-student`（多語情感分類），測試英文、日文句子的情感。
2. 試試 `pipeline("zero-shot-classification", model="facebook/bart-large-mnli")` 並搭配 `gr.Interface.from_pipeline()`，探索 Gradio 為這個 pipeline 生成什麼 UI。
3. 用 `gr.TabbedInterface` 把你自己完成的三個 Demo 合併成一個應用程式。

## 下一步

- `../02tokenizer/` — 深入了解 Tokenizer 的運作原理
- `../03Model/` — 直接操作模型 forward pass 與輸出解讀
- `../../05-Multimodal/` — 了解 AutoProcessor 如何統一處理文字與圖片輸入